In [4]:
import json
import pandas as pd
import os
from glob import glob

In [6]:
data = json.load(open('../data/processed_data/100_new.json',"r"))

In [8]:
data

{'0': {'sentence': '100\tPt, Tanecia, is a 4 year old female who is diagnosed with hypoplastic left heart syndrome s/p norwood, Glenn, and fenestrated fontan with a personal history of ECMO and seizure disorder who is now being referred for heart transplant psychosocial assessment.',
  'SDoH': [{'Healthcare': {'Experiencer': 'patients',
     'HealthcareType': 'diagnosis'}}]},
 '1': {'sentence': "Information for this assessment was obtained through an interview with the pt's mother and review of the medical records."},
 '2': {'sentence': 'SW spent 45 minutes of face to face time with patient and family.'},
 '3': {'sentence': 'Living situation:  Pt, Tanecia, lives with her parents and sisters in Land O Lakes, Florida.',
  'SDoH': [{'Living': {'Experiencer': 'patients',
     'LivingStatus': 'current',
     'LivingType': 'with both parents',
     'ResidentType': 'home'}}]},
 '4': {'sentence': 'They reside at 4427 Dylan Loop, # 187, Land O Lakes, FL  34639.',
  'SDoH': [{'Living': {'Experie

In [1]:
def normalize_event_for_csv(event_dict):
    """Normalizes the dictionary and returns a dictionary of standardized keys and values."""
    normalized = {}
    for key, value in event_dict.items():
        norm_key = key.lower().replace("_", "")
        if norm_key == "category":
            continue
        norm_val = str(value).lower().strip()
        normalized[norm_key] = norm_val
    return normalized

def calculate_tuple_overlap(p_dict, t_dict):
    """Calculates the overlap count between two normalized dictionaries."""
    overlap = 0
    for k, v in p_dict.items():
        if k in t_dict and t_dict[k] == v:
            overlap += 1
    return overlap

def generate_csv_rows(pred_events, true_events, category, sent_idx, sentence):
    """Aligns events and generates row dictionaries for the CSV."""
    rows = []
    
    pred_dicts = [normalize_event_for_csv(e) for e in pred_events]
    true_dicts = [normalize_event_for_csv(e) for e in true_events]
    
    matched_true_indices = set()
    
    # 1. Match predicted events to true events
    for p_dict in pred_dicts:
        best_match_idx = -1
        best_overlap = -1
        
        for j, t_dict in enumerate(true_dicts):
            if j in matched_true_indices:
                continue
            overlap = calculate_tuple_overlap(p_dict, t_dict)
            if overlap > best_overlap:
                best_overlap = overlap
                best_match_idx = j
                
        # If we found an alignment
        if best_match_idx != -1 and best_overlap > 0:
            matched_true_indices.add(best_match_idx)
            t_dict = true_dicts[best_match_idx]
            
            # Record True Positives and False Positives from the prediction
            for p_key, p_val in p_dict.items():
                if p_key in t_dict and t_dict[p_key] == p_val:
                    # It is a True Positive
                    rows.append([sent_idx, sentence, category, p_key, p_val, p_val, "TP"])
                else:
                    # The model hallucinated this specific argument
                    rows.append([sent_idx, sentence, category, p_key, None, p_val, "FP"])
                    
            # Record False Negatives (arguments the model missed in this matched event)
            for t_key, t_val in t_dict.items():
                if t_key not in p_dict or p_dict[t_key] != t_val:
                    rows.append([sent_idx, sentence, category, t_key, t_val, None, "FN"])
                    
        else:
            # Over-prediction (Entire event hallucinated)
            for p_key, p_val in p_dict.items():
                rows.append([sent_idx, sentence, category, p_key, None, p_val, "FP"])
                
    # 2. Handle Under-prediction (Entire event missed by the model)
    for j, t_dict in enumerate(true_dicts):
        if j not in matched_true_indices:
            for t_key, t_val in t_dict.items():
                rows.append([sent_idx, sentence, category, t_key, t_val, None, "FN"])
                
    return rows

def create_document_error_csv(doc_id, true_filepath, pred_filepath, output_dir):
    """Reads a single document pair and outputs the error analysis CSV."""
    with open(true_filepath, 'r', encoding='utf-8') as f:
        true_data = json.load(f)
    with open(pred_filepath, 'r', encoding='utf-8') as f:
        pred_data = json.load(f)
        
    all_csv_rows = []
    
    # Lookup table for predictions
    pred_lookup = {}
    for item in pred_data:
        sentence = item.get("sentence", "").strip()
        pred_lookup[sentence] = item.get("extracted_predictions", {})

    for idx, true_item in true_data.items():
        sentence = true_item.get("sentence", "").strip()
        true_sdoh_list = true_item.get("SDoH", [])
        pred_extracted = pred_lookup.get(sentence, {})
        
        all_categories = set()
        for event in true_sdoh_list:
            all_categories.update(event.keys())
        all_categories.update(pred_extracted.keys())
        
        for category in all_categories:
            true_events = [e[category] for e in true_sdoh_list if category in e]
            pred_events = pred_extracted.get(category, {}).get("extracted_conditions", [])
            
            # Generate the specific rows for this sentence and category
            sentence_rows = generate_csv_rows(pred_events, true_events, category, idx, sentence)
            all_csv_rows.extend(sentence_rows)

    # Convert to DataFrame and save
    if all_csv_rows:
        columns = ["sentence_index", "sentence", "category", "argument_key", "true_value", "pred_value", "error_type"]
        df = pd.DataFrame(all_csv_rows, columns=columns)
        
        # Ensure output directory exists
        os.makedirs(output_dir, exist_ok=True)
        
        output_path = os.path.join(output_dir, f"{doc_id}_error_analysis.csv")
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"Generated error analysis file: {output_path}")

# --- Example Usage ---
# create_document_error_csv("Doc_100", "data/true/100_true.json", "data/pred/100_pred.json", "output_csvs")

In [3]:
create_document_error_csv("Doc_100", "../data/processed_data/100_new.json", "../output/gpt4o/100_new_extracted.json", "../eval/100_new.csv")

Generated error analysis file: ../eval/100_new.csv\Doc_100_error_analysis.csv
